# Credit Risk Modeling: LightGBM (Gradient Boosted Trees)
## Project Overview
In this notebook, we build and train a **LightGBM (Light Gradient Boosting Machine)** model to predict loan default risk.

Gradient boosted decision trees handle non-linear relationships and high-order feature interactions automatically. LightGBM builds trees leaf-wise (best-first), which typically achieves higher accuracy and faster convergence than depth-wise algorithms on large tabular datasets.

### Pipeline Steps:
1. **Data Loading:** Ingest the raw loan dataset.
2. **Stratified Splitting:** Create stratified Train (64%), Validation (16%), and Test (20%) partitions.
3. **Feature Preprocessing:** Prepare continuous features with scaling and categorical features with target encoding via `ColumnTransformer`.
4. **LightGBM Model Training:** Configure hyperparameters (regularization, leaf limits, learning rate) and address class imbalance with `scale_pos_weight`.
5. **Experiment Tracking with MLflow:** Log evaluation metrics, diagnostic plots (ROC curve, PR curve, Confusion Matrix), and model artifacts.
6. **Inspection:** Review runs in the MLflow dashboard.

## 1. Initial Data Loading & Environment Setup
We load the loan performance dataset and identify numerical and categorical features.

In [ ]:
import kagglehub
import pandas
import glob
import os

# Download latest version
path = kagglehub.dataset_download("nikhil1e9/loan-default")
# Joins the folder path with the wildcard pattern
csv_files = glob.glob(os.path.join(path, "*.csv"))

print("Path to dataset files:", path)
print(csv_files)

df = pandas.read_csv(csv_files[0])
categorical_columns = (df.select_dtypes(include=['str']) if pandas.__version__[0] >= '3' else df.select_dtypes(include=['object'])).columns[1:].tolist()
print(categorical_columns)
df

['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,8C6S86ESGC,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,No,No,Other,No,0
255343,98R4KDHNND,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,No,No,Home,No,1
255344,XQK1UUUNGP,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,Yes,Yes,Auto,Yes,0
255345,JAO28CPL4H,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,Yes,Yes,Other,No,0


## 2. Stratified Data Splitting (Train, Validation, Test)
To evaluate our model reliably and prevent data leakage, we partition the dataset using **stratified splits**:

* **Test Set (20%):** Held out as a final unseen test set to evaluate real-world generalization.
* **Validation Set (16% of total):** Used for early stopping and hyperparameter monitoring during tree boosting.
* **Training Set (64% of total):** Used exclusively to fit the gradient boosted trees.

*Note: We stratify on both the default label (`Default`) and the composite categorical feature (`stratify_col`) to maintain consistent risk proportions across all subsets.*

In [3]:
from sklearn.model_selection import train_test_split


df['stratify_col'] = df[categorical_columns].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)

train_val_df_raw, test_df_raw = train_test_split(
    df, 
    test_size=0.20, 
    random_state=100, 
    stratify=df['stratify_col'] + '_' + df['Default'].astype(str) # <--- Stratifying on combined column
)

train_df_raw, val_df_raw = train_test_split(
    train_val_df_raw, 
    test_size=0.20, 
    random_state=100, 
    stratify=train_val_df_raw['stratify_col'] + '_' + train_val_df_raw['Default'].astype(str) # <--- Stratifying on combined column
)
del train_val_df_raw

print(f'Checking stratification: {train_df_raw["stratify_col"].unique().shape[0]} unique values in train set, {test_df_raw["stratify_col"].unique().shape[0]} unique values in test set')
print(f'Train size: {train_df_raw.shape[0]}, Validation size: {val_df_raw.shape[0]}, Test size: {test_df_raw.shape[0]}')

Checking stratification: 1920 unique values in train set, 1920 unique values in test set
Train size: 163421, Validation size: 40856, Test size: 51070


In [4]:
train_df = train_df_raw.copy()
val_df = val_df_raw.copy()
test_df = test_df_raw.copy()

In [5]:
X_train = train_df.drop(columns=['LoanID', 'Default'])
y_train = train_df['Default']
X_val = val_df.drop(columns=['LoanID', 'Default'])
y_val = val_df['Default']
X_test = test_df.drop(columns=['LoanID', 'Default'])
y_test = test_df['Default']

## 3. Feature Consideration: Loan-to-Income (LTI) Ratio
A standard financial metric in credit underwriting is the **Loan-to-Income (LTI) Ratio**:

$$\text{LTI Ratio} = \frac{\text{Loan Amount}}{\text{Annual Income}}$$

#### Context:
* **Standardized Leverage:** Normalizes loan size relative to annual earnings across different income tiers.
* **Tree Decision Boundaries:** Provides tree-based models with a direct leverage indicator for identifying high-risk thresholds.

In [ ]:
# def feature_engineering(df):
#     df['LTI'] = df['LoanAmount'] / df['Income']
#     return df

# X_train = feature_engineering(X_train)
# X_val = feature_engineering(X_val)
# X_test = feature_engineering(X_test)

## 4. LightGBM Model Training & MLflow Experiment Tracking
We build a Scikit-Learn `Pipeline` combining preprocessing with `LGBMClassifier` and track the experiment using `MLflowLogger` from `utils.py`.

### Architecture & Hyperparameter Design:
* **Preprocessing:** `ColumnTransformer` scales numerical features with `StandardScaler` and encodes categorical columns with `TargetEncoder`.
* **Overfitting Controls:** Moderate learning rate (`0.02`), constrained tree depth (`max_depth=5`), small leaf count (`num_leaves=31`), row subsampling (`subsample=0.8`), and L1/L2 regularization (`reg_alpha=0.5`, `reg_lambda=1.0`).
* **Imbalance Handling:** `scale_pos_weight=8.5` adjusts gradient updates to balance the ~11.6% minority default class against non-defaults.
* **Early Stopping:** Evaluates AUC on the validation set and stops training after 60 rounds without improvement.
* **Experiment Logging:** Records ROC-AUC, Gini, Precision, Recall, F1-Score, Brier Score, and diagnostic plots to MLflow.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, TargetEncoder
import lightgbm as lgb
from utils import MLflowLogger

drop_cols = ['stratify_col']

# Define which columns to scale and which to exclude
scale_cols = X_train.drop(columns=drop_cols).columns.difference(categorical_columns + ['stratify_col'])  # Assuming 'stratify_col_prob' is the only column we want to exclude from scaling


# Your pipeline
preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), scale_cols),
    ('target_encode', TargetEncoder(target_type='binary', random_state=42), categorical_columns),
], remainder='passthrough')

params = {
    "n_estimators": 800,
    "learning_rate": 0.04,
    "max_depth": 8,
    "num_leaves": 64,
    "scale_pos_weight": 8.2
}

# These parameters are tested to improve performance while preventing overfitting.
params = {
    'n_estimators': 1000,           # We will use early stopping
    'learning_rate': 0.02,          # Lower is better (0.01 - 0.05)
    'max_depth': 5,                 # Reduce from 8-10
    'num_leaves': 31,               # Very important - keep small (20-40)
    'min_child_samples': 50,        # Increase (prevents learning noise)
    'min_child_weight': 0.01,
    'subsample': 0.8,               # Row sampling
    'colsample_bytree': 0.7,        # Feature sampling
    'reg_alpha': 0.5,               # L1 regularization
    'reg_lambda': 1.0,              # L2 regularization
    'scale_pos_weight': 8.5,        # Keep according to your imbalance
    'random_state': 42,
    'verbose': -1,
    'objective': 'binary'
}

lgb_model = lgb.LGBMClassifier(**params)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', lgb_model)
])

logger = MLflowLogger(experiment_name="Credit_Risk_PD_Modeling")

final_model = logger.fit(
    model=pipeline,           # or raw model
    X_train=X_train.drop(columns=drop_cols),
    y_train=y_train,
    X_val=X_val.drop(columns=drop_cols),
    y_val=y_val,
    X_test=X_test.drop(columns=drop_cols),            # ← Optional but recommended
    y_test=y_test,
    run_name="LightGBM_with_LTI_scaled_improved_params",
    model_name="LightGBM",
    description="Initial LightGBM model with LTI and scaled numerical features with improved parameters.",
    # fit parameters for LightGBM...
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=60,
    eval_metric='auc'
)

Training LightGBM...


c:\Users\victo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   Train: AUC=0.7934 | Acc=0.6891 | F1=0.3596 | Prec=0.2364 | Rec=0.7516


c:\Users\victo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   Val: AUC=0.7541 | Acc=0.6773 | F1=0.3350 | Prec=0.2203 | Rec=0.6995


c:\Users\victo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/05/27 18:08:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


   Test: AUC=0.7479 | Acc=0.6768 | F1=0.3323 | Prec=0.2185 | Rec=0.6935
✅ Feature importance logged for train set


2026/05/27 18:08:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Run completed → LightGBM_with_LTI_scaled_improved_params



## 5. Accessing the MLflow Dashboard
Once model training completes, launch the MLflow user interface to inspect performance metrics and review generated diagnostic curves.

Run the following command in a terminal:
```bash
python -m mlflow ui --host 127.0.0.1 --port 5000
```
Then navigate to **[http://127.0.0.1:5000](http://127.0.0.1:5000)** in your browser.